In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load the pre-trained model
pretrained_model = tf.keras.models.load_model('best_model.keras')


In [ ]:

# 2. Freeze layers
for layer in pretrained_model.layers[:-2]:  # Freeze all layers except the last two
    layer.trainable = False


In [ ]:

# 3. Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()


In [ ]:

# Normalize and reshape the data
x_train = x_train.reshape((60000, 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((10000, 28, 28, 1)).astype('float32') / 255


In [ ]:

# 4. Modify the model for MNIST
model = models.Sequential(pretrained_model.layers[:-1])  # Remove the last layer
model.add(layers.Dense(10, activation='softmax'))  # Add new output layer for 10 classes


In [ ]:

# 5. Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:

# Print model summary
model.summary()


In [ ]:

# 6. Train the modified model
history = model.fit(x_train, y_train, 
                    epochs=5, 
                    batch_size=64,
                    validation_split=0.2,
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
                        tf.keras.callbacks.ModelCheckpoint('mnist_transfer_model.keras', save_best_only=True)
                    ])


In [ ]:

# 7. Evaluate the model
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f'\nTest accuracy: {test_acc}')


In [ ]:

# 8. Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title('Model Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Model Loss')

plt.show()